# Gold: patient_360

Builds `project1_gs_caresync.gold.patient_360` (one row per patient) from the silver layer per the STTM `sttm_silver_to_gold.xlsx`. Driving table: `silver.patients`; LEFT JOINs to `hospitals`, `insurance_providers` and patient-level lab_results aggregates. Full rebuild (CREATE OR REPLACE).

Schema notes: `patients.date_of_birth` is a timestamp in silver and is cast to `date`.

In [0]:
%sql
CREATE OR REPLACE TABLE project1_gs_caresync.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab test summary (STTM silver->gold)'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM project1_gs_caresync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                     AS patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                 AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                         AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                          AS insurance_provider_id,
  ip.provider_name                                                              AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                                      AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                              AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                              AS critical_test_count,
  la.last_test_date
FROM project1_gs_caresync.silver.patients p
LEFT JOIN project1_gs_caresync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN project1_gs_caresync.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id

In [0]:
%sql
-- Grain check: row count must equal silver.patients and patient_id must be unique
SELECT
  (SELECT COUNT(*) FROM project1_gs_caresync.silver.patients) AS silver_patients,
  COUNT(*)                                                    AS gold_rows,
  COUNT(DISTINCT patient_id)                                  AS distinct_patient_ids,
  SUM(total_tests)                                            AS sum_total_tests,
  (SELECT COUNT(*) FROM project1_gs_caresync.silver.lab_results) AS silver_lab_rows
FROM project1_gs_caresync.gold.patient_360

In [0]:
updated_df = spark.table("project1_gs_caresync.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))